# 01 — Explore the atom table

Flat view of every annotated atom across the corpus + the cross-article
edges that fall out of shared tag intersection.

In [1]:
import sys
import os
from pathlib import Path

# Find project root by walking up looking for our loader module
def find_project_root(marker='src/data/atom_table.py'):
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError(f"Could not find project root (no {marker} in any parent)")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 220)

from src.data.atom_table import (
    build_atom_table, summary_stats, tag_frequencies, show_atom,
    build_edge_table, add_edge_summary, article_connectivity,
    compute_tag_weights, show_tag_weights,
    compute_tag_similarities, build_semantic_edge_table, unique_tags_by_field,
    LIST_FIELDS,
)

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\gebruiker\Documents\Law & Tech


## 1. Load the atom table

In [2]:
df = build_atom_table()
print(f'{len(df)} atoms across {df["article_id"].nunique()} articles')
df.head()

49 atoms across 17 articles


,atom_id,article_id,lid,condition_type,text,text_nl,actors,legal_relations,acts,geographical_domain,temporal,explicit_references,hierarchies,residual,source,annotator,annotated_at,notes
0,7:454(1).a,7:454,1,post_condition,The care provider sets up a file relating to the treatment of the ...,De hulpverlener richt een dossier in met betrekking tot de behande...,"[care provider, patient]",[treatment agreement],[sets up],[],[],[],[],"[file, treatment]","Dutch Civil Code, Book 7, Title 7.7.5",Claude-draft,2026-06-22,Patient tagged as essential non-acting actor per schema. Treatment...
1,7:454(1).b,7:454,1,post_condition,He keeps a record in the file of data concerning the health of the...,Hij houdt in het dossier aantekening van de gegevens omtrent de ge...,"[care provider, patient]",[],[keeps record],[],[],[],[],"[file, data, health]","Dutch Civil Code, Book 7, Title 7.7.5",Claude-draft,2026-06-22,Pronoun 'He' resolves to care provider. Treatment agreement droppe...
2,7:454(1).c,7:454,1,post_condition,and the interventions performed with respect to them,en de te diens aanzien uitgevoerde verrichtingen,"[care provider, patient]",[],[keeps record],[],[],[],[],[interventions],"Dutch Civil Code, Book 7, Title 7.7.5",Claude-draft,2026-06-22,Pronoun 'them' resolves to patient. Same act-verb as .b applied to...
3,7:454(1).d,7:454,1,post_condition,and includes therein other documents containing such data,"en neemt andere stukken, bevattende zodanige gegevens, daarin op",[care provider],[],[includes],[],[],[],[],"[documents, data, file]","Dutch Civil Code, Book 7, Title 7.7.5",Claude-draft,2026-06-22,'Therein' refers to the file (residual). 'and' preserved per schem...
4,7:454(1).e,7:454,1,pre_condition,all this insofar as it is necessary for proper care to the patient,een en ander voor zover dit voor een goede hulpverlening aan de pa...,[patient],[],[],[],[],[],[],"[proper care, necessary]","Dutch Civil Code, Book 7, Title 7.7.5",Claude-draft,2026-06-22,Pre-condition (qualifier) limiting all duties in atoms .a-.d. 'Pro...


 ## 2. Per-article summary

In [3]:
summary_stats(df)

,article_id,n_atoms,n_pre,n_post,actors,legal_relations,acts,geographical_domain,temporal,explicit_references,hierarchies,residual
0,7:454,8,2,6,7,2,6,0,3,1,1,8
1,7:455,3,2,1,2,1,1,0,1,1,2,3
2,synth:1,2,1,1,2,0,2,2,1,0,0,1
3,synth:10,3,2,1,2,0,3,0,0,0,1,3
4,synth:11,3,2,1,3,1,2,0,1,0,0,3
5,synth:12,2,1,1,1,0,2,1,1,0,0,2
6,synth:13,3,2,1,2,0,2,1,1,0,0,3
7,synth:14,2,1,1,1,0,2,1,1,0,0,2
8,synth:15,3,2,1,3,0,3,0,0,0,0,3
9,synth:2,2,1,1,1,0,1,0,1,0,0,2


## 3. Tag frequencies

In [29]:
for field in LIST_FIELDS:
    freqs = tag_frequencies(df, field)
    if len(freqs) == 0:
        continue
    print(f'\n--- {field} ---')
    print(freqs.to_string())


--- actors ---
actors
care provider                 7
patient                       7
someone other than patient    1

--- legal_relations ---
legal_relations
treatment agreement    3

--- acts ---
acts
keeps record    2
sets up         1
includes        1
adds to         1
keeps           1
destroys        1

--- temporal ---
temporal
upon request                          1
for twenty years                      1
from the time of last modification    1
longer than twenty years              1
after request                         1

--- explicit_references ---
explicit_references
article 455    1
paragraph 1    1

--- hierarchies ---
hierarchies
Without prejudice to the provisions of article 455    1
Paragraph 1 does not apply insofar as                 1
nor insofar as                                        1

--- residual ---
residual
file                        6
data                        4
request                     3
treatment                   1
interventions               1


## 4. Build the edge table

One row per cross-article edge. Each edge is a single overlapping tag
value between two atoms in different articles.

In [30]:
edges = build_edge_table(df)
print(f'{len(edges)} cross-article edges')
edges

22 cross-article edges


,atom_a,atom_b,article_a,article_b,field,shared,strength,weighted_strength
0,7:454(1).a,7:455(1).a,7:454,7:455,actors,"[care provider, patient]",2,2.0
1,7:454(1).a,7:455(1).a,7:454,7:455,legal_relations,[treatment agreement],1,1.0
2,7:454(1).a,7:455(1).a,7:454,7:455,residual,[file],1,1.0
3,7:454(1).a,7:455(2).a,7:454,7:455,actors,[patient],1,1.0
4,7:454(1).b,7:455(1).a,7:454,7:455,actors,"[care provider, patient]",2,2.0
5,7:454(1).b,7:455(1).a,7:454,7:455,residual,"[data, file]",2,2.0
6,7:454(1).b,7:455(2).a,7:454,7:455,actors,[patient],1,1.0
7,7:454(1).b,7:455(2).a,7:454,7:455,residual,[data],1,1.0
8,7:454(1).c,7:455(1).a,7:454,7:455,actors,"[care provider, patient]",2,2.0
9,7:454(1).c,7:455(2).a,7:454,7:455,actors,[patient],1,1.0


## 5. Atom table with edge summary

Each atom now carries three extra columns: how many edges it has,
which atoms it's connected to, and which schema fields produce those
edges. This is the view where you can see *hubs* (high `n_edges`)
and *isolates* (`n_edges == 0`).

In [31]:
df_e = add_edge_summary(df, edges)
df_e[['atom_id', 'condition_type', 'n_edges', 'connected_atoms', 'edge_fields']]

,atom_id,condition_type,n_edges,connected_atoms,edge_fields
0,7:454(1).a,post_condition,4,"[7:455(1).a, 7:455(2).a]","[actors, legal_relations, residual]"
1,7:454(1).b,post_condition,4,"[7:455(1).a, 7:455(2).a]","[actors, residual]"
2,7:454(1).c,post_condition,2,"[7:455(1).a, 7:455(2).a]",[actors]
3,7:454(1).d,post_condition,3,"[7:455(1).a, 7:455(2).a]","[actors, residual]"
4,7:454(1).e,pre_condition,2,"[7:455(1).a, 7:455(2).a]",[actors]
5,7:454(2).a,post_condition,5,"[7:455(1).a, 7:455(2).a]","[actors, legal_relations, residual]"
6,7:454(3).a,post_condition,2,[7:455(1).a],"[actors, residual]"
7,7:454(3).b,pre_condition,0,[],[]
8,7:455(1).a,post_condition,14,"[7:454(1).a, 7:454(1).b, 7:454(1).c, 7:454(1).d, 7:454(1).e, 7:454...","[actors, legal_relations, residual]"
9,7:455(2).a,pre_condition,8,"[7:454(1).a, 7:454(1).b, 7:454(1).c, 7:454(1).d, 7:454(1).e, 7:454...","[actors, residual]"


In [32]:
# Sort by connectivity to find the hubs
df_e.sort_values('n_edges', ascending=False)[['atom_id', 'n_edges', 'connected_atoms']]

,atom_id,n_edges,connected_atoms
8,7:455(1).a,14,"[7:454(1).a, 7:454(1).b, 7:454(1).c, 7:454(1).d, 7:454(1).e, 7:454..."
9,7:455(2).a,8,"[7:454(1).a, 7:454(1).b, 7:454(1).c, 7:454(1).d, 7:454(1).e, 7:454..."
5,7:454(2).a,5,"[7:455(1).a, 7:455(2).a]"
0,7:454(1).a,4,"[7:455(1).a, 7:455(2).a]"
1,7:454(1).b,4,"[7:455(1).a, 7:455(2).a]"
3,7:454(1).d,3,"[7:455(1).a, 7:455(2).a]"
2,7:454(1).c,2,"[7:455(1).a, 7:455(2).a]"
6,7:454(3).a,2,[7:455(1).a]
4,7:454(1).e,2,"[7:455(1).a, 7:455(2).a]"
7,7:454(3).b,0,[]


In [33]:
# Isolated atoms — no cross-article edges yet
df_e.loc[df_e['n_edges'] == 0, ['atom_id', 'condition_type', 'text']]

,atom_id,condition_type,text
7,7:454(3).b,pre_condition,or as much longer as reasonably follows from the duty of a good ca...
10,7:455(2).b,pre_condition,nor insofar as any provision based on or by virtue of the law oppo...


## 6. Article-level connectivity

Aggregate edges to article we shud replace this with the graph after

In [34]:
article_connectivity(edges)

,article_a,article_b,n_edges,fields
0,7:454,7:455,22,"[actors, legal_relations, residual]"


## 7. Filter examples

In [35]:
# Which atoms share 'dossier' as a residual concept?
edges[edges['shared'].apply(lambda s: 'dossier' in s)]

,atom_a,atom_b,article_a,article_b,field,shared,strength,weighted_strength


In [36]:
# Edges by field — which channel produces the most edges? Right now its residual which is worrying
edges['field'].value_counts()

field
actors             12
residual            8
legal_relations     2
Name: count, dtype: int64

## 8. Tag weighting (discriminative weights)

Common tags like `hulpverlener` (in 7 of 11 atoms) don't really distinguish
anything. IDF down-weights them and up-weights rare/discriminative tags.

Two modes:
- `'idf'` — rare tags get high weight (log inverse document frequency)
- `'tf'` — common tags get high weight (raw frequency)


In [37]:
# Compute IDF weights for every (field, tag) pair
weights_idf = compute_tag_weights(df, method='idf')
print(f'Computed {len(weights_idf)} tag weights')

# View all weights sorted high to low
show_tag_weights(weights_idf)


Computed 42 tag weights


,field,tag,weight
0,acts,sets up,1.7918
1,actors,someone other than patient,1.7918
2,acts,keeps,1.7918
3,acts,destroys,1.7918
4,acts,adds to,1.7918
5,acts,includes,1.7918
6,temporal,longer than twenty years,1.7918
7,temporal,from the time of last modification,1.7918
8,temporal,for twenty years,1.7918
9,temporal,upon request,1.7918


In [38]:
# The most common tags get the lowest weights:
show_tag_weights(weights_idf).tail(8)


,field,tag,weight
34,residual,provision,1.7918
35,acts,keeps record,1.3863
36,legal_relations,treatment agreement,1.0986
37,residual,request,1.0986
38,residual,data,0.8755
39,residual,file,0.5390
40,actors,care provider,0.4055
41,actors,patient,0.4055


### Weighted edges — re-ranks the same 16 edges by discriminativeness

In [39]:
# Same engine, now passing weights
edges_weighted = build_edge_table(df, weights=weights_idf)
edges_weighted[['atom_a', 'atom_b', 'field', 'shared', 'strength', 'weighted_strength']]\
    .sort_values('weighted_strength', ascending=False)


,atom_a,atom_b,field,shared,strength,weighted_strength
17,7:454(2).a,7:455(1).a,residual,"[file, request]",2,1.6376
5,7:454(1).b,7:455(1).a,residual,"[data, file]",2,1.4145
11,7:454(1).d,7:455(1).a,residual,"[data, file]",2,1.4145
16,7:454(2).a,7:455(1).a,legal_relations,[treatment agreement],1,1.0986
19,7:454(2).a,7:455(2).a,residual,[request],1,1.0986
1,7:454(1).a,7:455(1).a,legal_relations,[treatment agreement],1,1.0986
12,7:454(1).d,7:455(2).a,residual,[data],1,0.8755
7,7:454(1).b,7:455(2).a,residual,[data],1,0.8755
4,7:454(1).b,7:455(1).a,actors,"[care provider, patient]",2,0.8109
0,7:454(1).a,7:455(1).a,actors,"[care provider, patient]",2,0.8109


### What changed?

The same edges as before, but `weighted_strength` is now informative:

- **`hulpverlener`-only edges drop to the bottom** (weight 0.41). They were 5 of the 16 edges and previously indistinguishable by `strength=1`.


This ranking is what cluster-detection algorithms will use later: weighted edges produce more meaningful cluster boundaries than uniform edges.


### Compare uniform vs weighted ranking side by side

In [40]:
# Top 5 edges by each ranking
print('--- TOP 5 BY UNIFORM STRENGTH ---')
print(edges_weighted.sort_values('strength', ascending=False).head(5)[['atom_a', 'atom_b', 'shared', 'strength']].to_string(index=False))
print()
print('--- TOP 5 BY WEIGHTED STRENGTH (IDF) ---')
print(edges_weighted.sort_values('weighted_strength', ascending=False).head(5)[['atom_a', 'atom_b', 'shared', 'weighted_strength']].to_string(index=False))


--- TOP 5 BY UNIFORM STRENGTH ---
    atom_a     atom_b                   shared  strength
7:454(1).a 7:455(1).a [care provider, patient]         2
7:454(1).b 7:455(1).a             [data, file]         2
7:454(1).b 7:455(1).a [care provider, patient]         2
7:454(2).a 7:455(1).a          [file, request]         2
7:454(1).d 7:455(1).a             [data, file]         2

--- TOP 5 BY WEIGHTED STRENGTH (IDF) ---
    atom_a     atom_b                shared  weighted_strength
7:454(2).a 7:455(1).a       [file, request]             1.6376
7:454(1).b 7:455(1).a          [data, file]             1.4145
7:454(1).d 7:455(1).a          [data, file]             1.4145
7:454(2).a 7:455(1).a [treatment agreement]             1.0986
7:454(2).a 7:455(2).a             [request]             1.0986


## 9. Semantic similarity (the new edge channel)

Exact set intersection misses synonyms and related concepts: `dossier` won't
match `gegevens`, `hulpverlener` won't match `zorgverlener`. The semantic
layer fixes that with a multilingual sentence-transformer.

**First run will download the model (~120 MB).** Subsequent runs use the
local cache. If you want a stronger model, swap the `model_name` argument
for `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` (~500 MB).


In [41]:
 # Make sure sentence-transformers is installed:
#   pip install sentence-transformers
# Then load the model once (cached locally after first download)
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
print('Model loaded.')


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded.


### Tag-level similarities

For every unique tag value in the corpus, compute cosine similarity with
every other tag in the same field. Keep pairs above the threshold.

In [42]:
sim_pairs = compute_tag_similarities(df, threshold=0.75, model=model)
print(f'{len(sim_pairs)} high-similarity tag pairs')
sim_pairs


0 high-similarity tag pairs


,field,tag_a,tag_b,similarity


### Atom-level semantic edges

Promote tag-level similarities to atom-level edges: if atom A has tag X and atom B has tag Y in the same field, and X~Y is above threshold, then A and B have a semantic edge in that field.

In [43]:
sem_edges = build_semantic_edge_table(df, sim_pairs)
print(f'{len(sem_edges)} semantic edges across atoms')
sem_edges[['atom_a', 'atom_b', 'field', 'matched_pairs', 'n_matches', 'max_similarity']]


0 semantic edges across atoms


KeyError: "None of [Index(['atom_a', 'atom_b', 'field', 'matched_pairs', 'n_matches', 'max_similarity'], dtype='object')] are in the [columns]"

### Combined view — exact edges vs semantic edges

Both channels operate on the same atoms. Exact edges are deterministic and auditable; semantic edges catch synonyms exact-match misses. Use both.

In [20]:
# Cross-article exact edges from earlier
exact = edges.copy()
exact['edge_kind'] = 'exact'

# Cross-article semantic edges, harmonised to the same columns
sem = sem_edges.copy()
sem['edge_kind'] = 'semantic'
sem['shared'] = sem['matched_pairs']
sem['strength'] = sem['n_matches']
sem['weighted_strength'] = sem['max_similarity']

cols = ['edge_kind', 'atom_a', 'atom_b', 'field', 'shared', 'strength', 'weighted_strength']
combined = pd.concat([exact[cols], sem[cols]], ignore_index=True)
combined.sort_values(['atom_a', 'atom_b', 'edge_kind'])


,edge_kind,atom_a,atom_b,field,shared,strength,weighted_strength
0,exact,7:454(1).a,7:455(1).a,actors,"[hulpverlener, patiënt]",2,2.0000
1,exact,7:454(1).a,7:455(1).a,legal_relations,[behandelingsovereenkomst],1,1.0000
2,exact,7:454(1).a,7:455(1).a,residual,[dossier],1,1.0000
15,semantic,7:454(1).a,7:455(2).a,residual,"[(behandeling, patiënt, 0.7575)]",1,0.7575
3,exact,7:454(1).b,7:455(1).a,actors,[hulpverlener],1,1.0000
4,exact,7:454(1).b,7:455(2).a,residual,[patiënt],1,1.0000
5,exact,7:454(1).c,7:455(1).a,actors,[hulpverlener],1,1.0000
6,exact,7:454(1).d,7:455(1).a,actors,[hulpverlener],1,1.0000
7,exact,7:454(1).d,7:455(1).a,residual,[gegevens],1,1.0000
8,exact,7:454(1).d,7:455(2).a,residual,[gegevens],1,1.0000


### How much extra coverage does the semantic layer add?

In [21]:
print(f'Exact edges:     {len(exact)}')
print(f'Semantic edges:  {len(sem_edges)}')

# Which atom pairs only connect via semantic edges, not exact?
exact_pairs = {(r.atom_a, r.atom_b) for _, r in exact.iterrows()}
sem_pairs   = {(r.atom_a, r.atom_b) for _, r in sem.iterrows()}
sem_only    = sem_pairs - exact_pairs
exact_only  = exact_pairs - sem_pairs
both        = sem_pairs & exact_pairs
print(f'Atom pairs connected by exact only:    {len(exact_only)}')
print(f'Atom pairs connected by semantic only: {len(sem_only)}')
print(f'Atom pairs connected by both:          {len(both)}')

if sem_only:
    print()
    print('Atom pairs ONLY connected semantically:')
    for ap in sorted(sem_only):
        print(f'  {ap[0]} <-> {ap[1]}')


Exact edges:     15
Semantic edges:  4
Atom pairs connected by exact only:    8
Atom pairs connected by semantic only: 3
Atom pairs connected by both:          1

Atom pairs ONLY connected semantically:
  7:454(1).a <-> 7:455(2).a
  7:454(1).e <-> 7:455(1).a
  7:454(3).b <-> 7:455(2).a
